In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table(
    "ecommerce_data_analysis.silver.silver_sales_enriched"
)

print("Silver loaded successfully")
print("Rows:", silver.count())
print("Columns:", len(silver.columns))

Silver loaded successfully
Rows: 250000
Columns: 32


1.Total sales : Gross Revenue + units sold + AOV

In [0]:
from pyspark.sql import functions as F

total_sales = silver.agg(
    F.round(F.sum("Total_Amount"), 2).alias("Gross_Revenue"),
    F.sum("Quantity").alias("Total_Units_Sold"),
    F.round(
        F.sum("Total_Amount") / F.countDistinct("Order_ID"), 2
    ).alias("AOV")
)

display(total_sales)

Gross_Revenue,Total_Units_Sold,AOV
5.9306932543E9,312395,23722.77


2.Monthly Revenue 

In [0]:
monthly_revenue = (
    silver
    .withColumn("Year", F.year("Order_Date"))
    .withColumn("Month", F.month("Order_Date"))
    .groupBy("Year", "Month")
    .agg(
        F.round(F.sum("Total_Amount"), 2).alias("Gross_Revenue")
    )
    .orderBy("Year", "Month")
)

display(monthly_revenue)

Year,Month,Gross_Revenue
2024,6,2.4078169745E8
2024,7,2.4096625962E8
2024,8,2.4019462437E8
2024,9,2.29331306E8
2024,10,2.3997159664E8
2024,11,2.3455779966E8
2024,12,2.4195101768E8
2025,1,2.4122915524E8
2025,2,2.151931355E8
2025,3,2.4263111034E8


3.Revenue Trend / Quarterly Revenue

In [0]:
quarterly_revenue = (
    silver
    .withColumn("Year", F.year("Order_Date"))
    .withColumn("Quarter", F.quarter("Order_Date"))
    .groupBy("Year", "Quarter")
    .agg(
        F.round(F.sum("Total_Amount"), 2).alias("Gross_Revenue"),
        F.sum("Quantity").alias("Units_Sold"),
        F.countDistinct("Order_ID").alias("Total_Orders")
    )
    .orderBy("Year", "Quarter")
)

display(quarterly_revenue)

Year,Quarter,Gross_Revenue,Units_Sold,Total_Orders
2024,2,2.4078169745E8,12532,10008
2024,3,7.1049218999E8,37533,30078
2024,4,7.1648041398E8,37775,30055
2025,1,6.9905340108E8,37121,29750
2025,2,7.1489930281E8,37695,30179
2025,3,7.1226999588E8,37602,30151
2025,4,7.2368908085E8,38018,30439
2026,1,7.1012998084E8,37076,29600
2026,2,7.0289719142E8,37043,29740


Month-to-Month change

In [0]:
from pyspark.sql.window import Window

w = Window.orderBy("Year", "Month")

revenue_trend = (
    monthly_revenue
    .withColumn(
        "Previous_Revenue",
        F.lag("Gross_Revenue").over(w)
    )
    .withColumn(
        "Revenue_Change_Percent",
        F.round(
            (
                (F.col("Gross_Revenue") - F.col("Previous_Revenue"))
                / F.col("Previous_Revenue")
            ) * 100,
            2
        )
    )
)

display(revenue_trend)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Year,Month,Gross_Revenue,Previous_Revenue,Revenue_Change_Percent
2024,6,2.4078169745E8,null,null
2024,7,2.4096625962E8,2.4078169745E8,0.08
2024,8,2.4019462437E8,2.4096625962E8,-0.32
2024,9,2.29331306E8,2.4019462437E8,-4.52
2024,10,2.3997159664E8,2.29331306E8,4.64
2024,11,2.3455779966E8,2.3997159664E8,-2.26
2024,12,2.4195101768E8,2.3455779966E8,3.15
2025,1,2.4122915524E8,2.4195101768E8,-0.3
2025,2,2.151931355E8,2.4122915524E8,-10.79
2025,3,2.4263111034E8,2.151931355E8,12.75


4. Order Status Ananlysis

In [0]:
order_status = (
    silver
    .groupBy("Order_Status")
    .agg(
        F.countDistinct("Order_ID").alias("Total_Orders")
    )
)

total_orders = silver.select(
    F.countDistinct("Order_ID").alias("Total")
).collect()[0]["Total"]

order_status = order_status.withColumn(
    "Percentage",
    F.round(
        F.col("Total_Orders") / F.lit(total_orders) * 100,
        2
    )
)

display(order_status)

Order_Status,Total_Orders,Percentage
DELIVERED,200139,80.06
PROCESSING,12402,4.96
RETURNED,12493,5.0
CANCELLED,12507,5.0
SHIPPED,12459,4.98


Monthly Failure rate

In [0]:
monthly_failure = (
    silver
    .withColumn("Year", F.year("Order_Date"))
    .withColumn("Month", F.month("Order_Date"))
    .groupBy("Year", "Month")
    .agg(
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.countDistinct(
            F.when(
                F.lower(F.col("Order_Status")).isin(
                    "cancelled", "canceled", "returned"
                ),
                F.col("Order_ID")
            )
        ).alias("Failed_Orders")
    )
    .withColumn(
        "Failure_Rate_Percent",
        F.round(
            F.col("Failed_Orders") /
            F.col("Total_Orders") * 100,
            2
        )
    )
    .orderBy("Year", "Month")
)

display(monthly_failure)

Year,Month,Total_Orders,Failed_Orders,Failure_Rate_Percent
2024,6,10008,1041,10.4
2024,7,10216,1050,10.28
2024,8,10158,1066,10.49
2024,9,9704,940,9.69
2024,10,10034,1049,10.45
2024,11,9865,1040,10.54
2024,12,10156,1032,10.16
2025,1,10243,1007,9.83
2025,2,9211,929,10.09
2025,3,10296,1014,9.85


5. Payment Method Analysis

In [0]:
payment_analysis = (
    silver
    .groupBy("Payment_Mode")
    .agg(
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Sales"),
        F.round(F.avg("Total_Amount"), 2).alias("Average_Transaction")
    )
    .orderBy(F.desc("Total_Sales"))
)

display(payment_analysis)

Payment_Mode,Total_Orders,Total_Sales,Average_Transaction
UPI,128474,3.17641692696E9,24724.2
COD,82093,1.8419087552E9,22436.86
DEBIT CARD,28856,6.3170758446E8,21891.72
CREDIT CARD,10577,2.8065998768E8,26534.93


Sales percentage by payment mode

In [0]:
payment_total = silver.agg(
    F.sum("Total_Amount").alias("Total")
).collect()[0]["Total"]

payment_analysis = payment_analysis.withColumn(
    "Sales_Percentage",
    F.round(
        F.col("Total_Sales") / F.lit(payment_total) * 100,
        2
    )
)

display(payment_analysis)

Payment_Mode,Total_Orders,Total_Sales,Average_Transaction,Sales_Percentage
UPI,128474,3.17641692696E9,24724.2,53.56
COD,82093,1.8419087552E9,22436.86,31.06
DEBIT CARD,28856,6.3170758446E8,21891.72,10.65
CREDIT CARD,10577,2.8065998768E8,26534.93,4.73


6. State-wise Sales

In [0]:
state_sales = (
    silver
    .groupBy("State")
    .agg(
        F.round(F.sum("Total_Amount"), 2).alias("Gross_Revenue"),
        F.sum("Quantity").alias("Units_Sold"),
        F.countDistinct("Order_ID").alias("Total_Orders")
    )
    .orderBy(F.desc("Gross_Revenue"))
)

display(state_sales)

State,Gross_Revenue,Units_Sold,Total_Orders
UP,7.6835100766E8,40763,32631
Rajasthan,7.5728646133E8,39564,31666
Haryana,7.5521523875E8,40122,32064
Maharashtra,6.0583068657E8,31905,25478
Punjab,5.9922541049E8,31482,25235
Gujarat,5.9917774127E8,31467,25171
Delhi,4.7446922648E8,24354,19524
Tamil Nadu,4.7051570659E8,24832,19792
West Bengal,4.563199865E8,24283,19450
Karnataka,4.4430178866E8,23623,18989


Revenue Percentage

In [0]:
state_total = silver.agg(
    F.sum("Total_Amount").alias("Total")
).collect()[0]["Total"]

state_sales = state_sales.withColumn(
    "Revenue_Percentage",
    F.round(
        F.col("Gross_Revenue") / F.lit(state_total) * 100,
        2
    )
)

display(state_sales)

State,Gross_Revenue,Units_Sold,Total_Orders,Revenue_Percentage
UP,7.6835100766E8,40763,32631,12.96
Rajasthan,7.5728646133E8,39564,31666,12.77
Haryana,7.5521523875E8,40122,32064,12.73
Maharashtra,6.0583068657E8,31905,25478,10.22
Punjab,5.9922541049E8,31482,25235,10.1
Gujarat,5.9917774127E8,31467,25171,10.1
Delhi,4.7446922648E8,24354,19524,8.0
Tamil Nadu,4.7051570659E8,24832,19792,7.93
West Bengal,4.563199865E8,24283,19450,7.69
Karnataka,4.4430178866E8,23623,18989,7.49


State cintributing to top 80% revenue

In [0]:
w = Window.orderBy(F.desc("Gross_Revenue"))

top_80_states = (
    state_sales
    .withColumn(
        "Cumulative_Revenue",
        F.sum("Revenue_Percentage").over(w)
    )
    .filter(F.col("Cumulative_Revenue") <= 80)
)

display(top_80_states)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


State,Gross_Revenue,Units_Sold,Total_Orders,Revenue_Percentage,Cumulative_Revenue
UP,7.6835100766E8,40763,32631,12.96,12.96
Rajasthan,7.5728646133E8,39564,31666,12.77,25.73
Haryana,7.5521523875E8,40122,32064,12.73,38.46
Maharashtra,6.0583068657E8,31905,25478,10.22,48.68
Punjab,5.9922541049E8,31482,25235,10.1,58.78
Gujarat,5.9917774127E8,31467,25171,10.1,68.88
Delhi,4.7446922648E8,24354,19524,8.0,76.88


7. City-Wise Sales

In [0]:
from pyspark.sql import functions as F

city_analysis = (
    silver
    .groupBy("Customer_Tier")
    .agg(
        F.countDistinct("Order_ID").alias("Order_Volume"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Sales"),
        F.round(
            F.sum("Total_Amount") / F.countDistinct("Order_ID"),
            2
        ).alias("Average_Basket_Size")
    )
    .orderBy("Customer_Tier")
)

display(city_analysis)

Customer_Tier,Order_Volume,Total_Sales,Average_Basket_Size
Gold,37065,4.0144558861E8,10830.85
Platinum,187576,5.34229273266E9,28480.68
Silver,25359,1.8695493303E8,7372.33


In [0]:
print("City_Tier" in silver.columns)

False


City Level Analysis

In [0]:
city_sales = (
    silver
    .groupBy("City")
    .agg(
        F.countDistinct("Order_ID").alias("Order_Volume"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Sales"),
        F.round(
            F.sum("Total_Amount") /
            F.countDistinct("Order_ID"),
            2
        ).alias("Average_Basket_Size")
    )
    .orderBy(F.desc("Total_Sales"))
)

display(city_sales)

City,Order_Volume,Total_Sales,Average_Basket_Size
Madurai,6946,1.6691152422E8,24029.88
New Delhi,6840,1.6679610919E8,24385.4
Howrah,6867,1.5970469467E8,23256.84
Kanpur,6644,1.5848019955E8,23853.13
Thane,6522,1.5767350788E8,24175.64
Varanasi,6762,1.5680673242E8,23189.4
Amritsar,6526,1.567499959E8,24019.31
Dwarka,6562,1.5663395706E8,23869.85
Chennai,6668,1.566299289E8,23489.79
Surat,6536,1.5659268146E8,23958.49


8 . Top 50 Customers

In [0]:
top_customers = (
    silver
    .groupBy("Customer_ID")
    .agg(
        F.round(F.sum("Total_Amount"), 2).alias("Total_Spending"),
        F.countDistinct("Order_ID").alias("Total_Orders")
    )
    .withColumn(
        "Average_Order_Value",
        F.round(
            F.col("Total_Spending") /
            F.col("Total_Orders"),
            2
        )
    )
    .orderBy(F.desc("Total_Spending"))
    .limit(50)
)

display(top_customers)

Customer_ID,Total_Spending,Total_Orders,Average_Order_Value
CUST00018065,1071444.02,10,107144.4
CUST00017168,947342.64,11,86122.06
CUST00000699,913494.32,10,91349.43
CUST00023830,898661.2,8,112332.65
CUST00004600,890131.55,9,98903.51
CUST00033067,888994.45,11,80817.68
CUST00020280,877895.43,7,125413.63
CUST00021066,876932.74,13,67456.36
CUST00003487,869402.68,11,79036.61
CUST00009202,843527.62,12,70293.97


9. Top 10 Best-Selling Products 

In [0]:
top_products = (
    silver
    .groupBy("Product_ID")
    .agg(
        F.sum("Quantity").alias("Total_Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue")
    )
    .orderBy(F.desc("Total_Units_Sold"))
    .limit(10)
)

display(top_products)

Product_ID,Total_Units_Sold,Total_Revenue
PROD000004,333,1.423097403E7
PROD001053,328,3197831.12
PROD000176,328,1.441490504E7
PROD000424,327,2.781628993E7
PROD000949,324,2.018639165E7
PROD001368,324,1.764392933E7
PROD001297,321,2.059677346E7
PROD001891,320,7481876.17
PROD001508,320,1.428347093E7
PROD001474,319,1.402903553E7


Single-Unit sales

In [0]:
single_unit_products = (
    silver
    .filter(F.col("Quantity") == 1)
    .groupBy("Product_ID")
    .agg(
        F.count("*").alias("Single_Unit_Sales"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue")
    )
    .orderBy(F.desc("Single_Unit_Sales"))
    .limit(10)
)

display(single_unit_products)

Product_ID,Single_Unit_Sales,Total_Revenue
PROD000739,228,1.236498322E7
PROD000949,217,1.353482542E7
PROD001297,215,1.379398263E7
PROD001508,213,9518980.73
PROD000316,211,2423372.41
PROD000004,211,9032114.12
PROD001929,210,1.668733922E7
PROD000530,209,7913284.55
PROD001787,208,1.110802662E7
PROD000565,208,1.023805152E7


10. Category Performance

In [0]:
category_analysis = (
    silver
    .groupBy("Category")
    .agg(
        F.sum("Quantity").alias("Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.round(
            F.sum("Total_Amount") /
            F.countDistinct("Order_ID"),
            2
        ).alias("Average_Order_Value")
    )
    .orderBy(F.desc("Total_Revenue"))
)

display(category_analysis)

Category,Units_Sold,Total_Revenue,Total_Orders,Average_Order_Value
Electronics,83333,4.30634999963E9,66783,64482.73
Home,42163,7.1612508265E8,33653,21279.68
Sports,36729,4.7583914504E8,29341,16217.55
Fashion,62327,3.3367879401E8,49802,6700.11
Grocery,41653,4.177139725E7,33402,1250.57
Beauty,20896,3.826580074E7,16724,2288.08
Books,25294,1.866303498E7,20295,919.59


11. Brand Performance

In [0]:
brand_analysis = (
    silver
    .groupBy("Brand")
    .agg(
        F.sum("Quantity").alias("Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue"),
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.round(F.avg("Total_Amount"), 2).alias("Average_Sale_Value")
    )
    .orderBy(F.desc("Total_Revenue"))
)

display(brand_analysis)

Brand,Units_Sold,Total_Revenue,Total_Orders,Average_Sale_Value
HP,16619,9.4214723912E8,13353,70556.97
Noise,16807,9.1426322056E8,13418,68137.07
boAt,16700,8.4248616545E8,13411,62820.53
Samsung,16557,8.3531656933E8,13280,62900.34
Apple,16650,7.7213680517E8,13321,57963.88
Nike,20155,1.7680764123E8,16005,11047.03
Puma,19909,1.5998031435E8,16000,9998.77
Urban,8366,1.5113325225E8,6664,22679.06
Godrej,8366,1.4998282419E8,6742,22246.04
Philips,8519,1.4381474616E8,6793,21171.02


Revenue percentage by brand

In [0]:
brand_total = silver.agg(
    F.sum("Total_Amount").alias("Total")
).collect()[0]["Total"]

brand_analysis = brand_analysis.withColumn(
    "Revenue_Percentage",
    F.round(
        F.col("Total_Revenue") / F.lit(brand_total) * 100,
        2
    )
)

display(brand_analysis)

Brand,Units_Sold,Total_Revenue,Total_Orders,Average_Sale_Value,Revenue_Percentage
HP,16619,9.4214723912E8,13353,70556.97,15.89
Noise,16807,9.1426322056E8,13418,68137.07,15.42
boAt,16700,8.4248616545E8,13411,62820.53,14.21
Samsung,16557,8.3531656933E8,13280,62900.34,14.08
Apple,16650,7.7213680517E8,13321,57963.88,13.02
Nike,20155,1.7680764123E8,16005,11047.03,2.98
Puma,19909,1.5998031435E8,16000,9998.77,2.7
Urban,8366,1.5113325225E8,6664,22679.06,2.55
Godrej,8366,1.4998282419E8,6742,22246.04,2.53
Philips,8519,1.4381474616E8,6793,21171.02,2.42


12. Product Ratings

In [0]:
rating_analysis = (
    silver
    .groupBy("Category")
    .agg(
        F.round(F.avg("Rating"), 2).alias("Average_Rating"),
        F.sum("Quantity").alias("Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue")
    )
    .orderBy(F.desc("Average_Rating"))
)

display(rating_analysis)

Category,Average_Rating,Units_Sold,Total_Revenue
Grocery,4.4,41653,4.177139725E7
Fashion,4.4,62327,3.3367879401E8
Electronics,4.4,83333,4.30634999963E9
Home,4.4,42163,7.1612508265E8
Beauty,4.4,20896,3.826580074E7
Books,4.39,25294,1.866303498E7
Sports,4.39,36729,4.7583914504E8


Rating v/s Sales

In [0]:
rating_sales = (
    silver
    .groupBy("Rating")
    .agg(
        F.sum("Quantity").alias("Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue")
    )
    .orderBy("Rating")
)

display(rating_sales)

Rating,Units_Sold,Total_Revenue
null,162502,3.09437829624E9
3.0,15082,2.8169572202E8
4.0,59688,1.13351899788E9
5.0,75123,1.42110023816E9


13. Coupon Performance

In [0]:
coupon_analysis = (
    silver
    .filter(F.col("Coupon_Code").isNotNull())
    .groupBy("Coupon_Code")
    .agg(
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.sum("Quantity").alias("Units_Sold"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue"),
        F.round(F.avg("Total_Amount"), 2).alias("Average_Order_Value")
    )
    .orderBy(F.desc("Total_Revenue"))
)

display(coupon_analysis)

Coupon_Code,Total_Orders,Units_Sold,Total_Revenue,Average_Order_Value
SAVE10,24997,31110,5.456185883E8,21827.36
DIWALI100,12609,15879,3.0772042554E8,24404.82
FLAT50,12579,15781,3.0389584032E8,24158.98


Coupon v/s noncoupon AOV

In [0]:
coupon_aov = (
    silver
    .withColumn(
        "Coupon_Status",
        F.when(
            F.col("Coupon_Code").isNull(),
            "No Coupon"
        ).otherwise("Coupon Used")
    )
    .groupBy("Coupon_Status")
    .agg(
        F.countDistinct("Order_ID").alias("Total_Orders"),
        F.round(F.sum("Total_Amount"), 2).alias("Total_Revenue"),
        F.round(
            F.sum("Total_Amount") /
            F.countDistinct("Order_ID"),
            2
        ).alias("AOV")
    )
)

display(coupon_aov)

Coupon_Status,Total_Orders,Total_Revenue,AOV
No Coupon,199815,4.77345840014E9,23889.39
Coupon Used,50185,1.15723485416E9,23059.38
